In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from sklearn.impute import SimpleImputer
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
from sksurv.util import Surv
from sklearn.model_selection import KFold
from sksurv.metrics import concordance_index_ipcw
from sksurv.ensemble import RandomSurvivalForest
import lightgbm as lgb

from dectections_karyotypes import cytogenetic_analysis, find_anomalies, detect_anomaly

import optuna

In [178]:
# Clinical Data
df = pd.read_csv("./clinical_train.csv")
df_eval = pd.read_csv("./clinical_test.csv")

# Molecular Data
maf_df = pd.read_csv("./molecular_train.csv")
maf_eval = pd.read_csv("./molecular_test.csv")

target_df = pd.read_csv("./target_train.csv")
# target_df_test = pd.read_csv("./target_test.csv")

target_df.dropna(subset=['OS_YEARS', 'OS_STATUS'], inplace=True)

# Preview the data
df.head()

,ID,CENTER,BM_BLAST,WBC,ANC,MONOCYTES,HB,PLT,CYTOGENETICS
0,P132697,MSK,14.0,2.8,0.2,0.7,7.6,119.0,"46,xy,del(20)(q12)[2]/46,xy[18]"
1,P132698,MSK,1.0,7.4,2.4,0.1,11.6,42.0,"46,xx"
2,P116889,MSK,15.0,3.7,2.1,0.1,14.2,81.0,"46,xy,t(3;3)(q25;q27)[8]/46,xy[12]"
3,P132699,MSK,1.0,3.9,1.9,0.1,8.9,77.0,"46,xy,del(3)(q26q27)[15]/46,xy[5]"
4,P132700,MSK,6.0,128.0,9.7,0.9,11.1,195.0,"46,xx,t(3;9)(p13;q22)[10]/46,xx[10]"


In [179]:
df_eval.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1193 entries, 0 to 1192
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ID            1193 non-null   object 
 1   CENTER        1193 non-null   object 
 2   BM_BLAST      1078 non-null   float64
 3   WBC           1081 non-null   float64
 4   ANC           1052 non-null   float64
 5   MONOCYTES     310 non-null    float64
 6   HB            1082 non-null   float64
 7   PLT           1078 non-null   float64
 8   CYTOGENETICS  1077 non-null   object 
dtypes: float64(6), object(3)
memory usage: 84.0+ KB


In [180]:
maf_eval.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3089 entries, 0 to 3088
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID              3089 non-null   object 
 1   CHR             3020 non-null   object 
 2   START           3020 non-null   float64
 3   END             3020 non-null   float64
 4   REF             3020 non-null   object 
 5   ALT             3020 non-null   object 
 6   GENE            3089 non-null   object 
 7   PROTEIN_CHANGE  3043 non-null   object 
 8   EFFECT          2999 non-null   object 
 9   VAF             3089 non-null   float64
 10  DEPTH           3020 non-null   float64
dtypes: float64(4), object(7)
memory usage: 265.6+ KB


In [181]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3323 entries, 0 to 3322
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ID            3323 non-null   object 
 1   CENTER        3323 non-null   object 
 2   BM_BLAST      3214 non-null   float64
 3   WBC           3051 non-null   float64
 4   ANC           3130 non-null   float64
 5   MONOCYTES     2722 non-null   float64
 6   HB            3213 non-null   float64
 7   PLT           3199 non-null   float64
 8   CYTOGENETICS  2936 non-null   object 
dtypes: float64(6), object(3)
memory usage: 233.8+ KB


In [182]:
df_representation = pd.merge(target_df, df, on = "ID", how = "left")

T = df_representation["OS_YEARS"]
E = df_representation["OS_STATUS"]

col_float = ["BM_BLAST", "WBC", "ANC", "MONOCYTES", "HB", "PLT"]
for col in col_float:

    mask_missing = df_representation[col].isna()
    
    results = logrank_test(T[~mask_missing], T[mask_missing],
                            event_observed_A=E[~mask_missing],
                            event_observed_B=E[mask_missing])

    print(f"p-value for {col}=", results.p_value)
    
    
print("\nSample size is small, so i don't value that much the p_value especially for BM_BLAST, but in case i'll add an indicator for BM_BLAST, ANC, MONOCYTES + imputation by median") 

p-value for BM_BLAST= 0.0306746738510227
p-value for WBC= 0.7115987056874484
p-value for ANC= 0.22029401507554724
p-value for MONOCYTES= 0.5182785200791407
p-value for HB= 0.9056092312235388
p-value for PLT= 0.9821735777356673

Sample size is small, so i don't value that much the p_value especially for BM_BLAST, but in case i'll add an indicator for BM_BLAST, ANC, MONOCYTES + imputation by median


In [183]:
# Missing values non floar

df["CYTOGENETICS"] = df["CYTOGENETICS"].fillna("unknown")

In [184]:
anomalies = find_anomalies(df, k = 50)

df_analysis = pd.merge(target_df, df[["ID","CYTOGENETICS"]], on = "ID", how = "left")
cytogenetic_analysis(df_analysis, anomalies, "OS_YEARS")[1].head()


,anomaly,p_value,present_cases
23,25,0.004634,161
21,33,0.068479,161
34,26,0.193345,86
33,24,0.431513,85
24,23,0.434610,122


In [185]:
# EVAL

df_analysis_eval = df_eval.copy()

for anomaly in anomalies:
        df_analysis_eval[anomaly] = df_eval["CYTOGENETICS"].apply(lambda x: detect_anomaly(str(x), anomaly))

In [186]:
data = df.copy()
data.drop("CYTOGENETICS", axis = 1, inplace = True)

# Imputation for missing float values

imputer = SimpleImputer(strategy = "median")
data[col_float] = imputer.fit_transform(data[col_float])
data[col_float] = imputer.transform(df[col_float])

# Indicators for suspects values

suspects = ["BM_BLAST", "ANC", "MONOCYTES"]

for s in suspects:
    data[f"NOT_MISS_{s}"] = df[s].notna().astype(int)


data["MALE"] = df["CYTOGENETICS"].astype(str).str.lower().str.startswith("46,xx").astype(int)
data["FEMALE"] = df["CYTOGENETICS"].astype(str).str.lower().str.startswith("46,xy").astype(int)
data["UNKNOWN_SEX"] = (df["CYTOGENETICS"] == "unknown").astype(int)

# We are adding features from Cytogenetics data, and fill NaN by 0 (choice)

data = pd.merge(data, df_analysis[["ID"] + anomalies], on = "ID", how = "left")
data.loc[:, anomalies] = data[anomalies].fillna(0)
data[anomalies] = data[anomalies].astype(int)

In [187]:
# EVAL

data_eval = df_eval.copy()
data_eval.drop("CYTOGENETICS", axis = 1, inplace = True)

# Imputation for missing float values

imputer = SimpleImputer(strategy = "median")
data_eval[col_float] = imputer.fit_transform(data_eval[col_float])
data_eval[col_float] = imputer.transform(df_eval[col_float])

# Indicators for suspects values

suspects = ["BM_BLAST", "ANC", "MONOCYTES"]

for s in suspects:
    data_eval[f"NOT_MISS_{s}"] = df_eval[s].notna().astype(int)


data_eval["MALE"] = df_eval["CYTOGENETICS"].astype(str).str.lower().str.startswith("46,xx").astype(int)
data_eval["FEMALE"] = df_eval["CYTOGENETICS"].astype(str).str.lower().str.startswith("46,xy").astype(int)
data_eval["UNKNOWN_SEX"] = (df_eval["CYTOGENETICS"] == "unknown").astype(int)

# We are adding features from Cytogenetics data, and fill NaN by 0 (choice)

data_eval = pd.merge(data_eval, df_analysis_eval[["ID"] + anomalies], on = "ID", how = "left")
data_eval.loc[:, anomalies] = data_eval[anomalies].fillna(0)
data_eval[anomalies] = data_eval[anomalies].astype(int)

# Adding molecular data

In [188]:
maf_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10935 entries, 0 to 10934
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID              10935 non-null  object 
 1   CHR             10821 non-null  object 
 2   START           10821 non-null  float64
 3   END             10821 non-null  float64
 4   REF             10821 non-null  object 
 5   ALT             10821 non-null  object 
 6   GENE            10935 non-null  object 
 7   PROTEIN_CHANGE  10923 non-null  object 
 8   EFFECT          10935 non-null  object 
 9   VAF             10846 non-null  float64
 10  DEPTH           10821 non-null  float64
dtypes: float64(4), object(7)
memory usage: 939.9+ KB


In [189]:
maf_df.head()

,ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH
0,P100000,11,119149248.0,119149248.0,G,A,CBL,p.C419Y,non_synonymous_codon,0.0830,1308.0
1,P100000,5,131822301.0,131822301.0,G,T,IRF1,p.Y164*,stop_gained,0.0220,532.0
2,P100000,3,77694060.0,77694060.0,G,C,ROBO2,p.?,splice_site_variant,0.4100,876.0
3,P100000,4,106164917.0,106164917.0,G,T,TET2,p.R1262L,non_synonymous_codon,0.4300,826.0
4,P100000,2,25468147.0,25468163.0,ACGAAGAGGGGGTGTTC,A,DNMT3A,p.E505fs*141,frameshift_variant,0.0898,942.0


In [190]:
seuil = 10
GENE =  pd.crosstab(maf_df["ID"], maf_df["GENE"])
GENE = GENE.loc[:, GENE.sum(axis=0) > seuil]

data = pd.merge(data, GENE, on = "ID", how = "left")

In [191]:
# GENE EVAL

GENE_eval =  pd.crosstab(maf_eval["ID"], maf_eval["GENE"])
GENE_eval = GENE_eval.loc[:, GENE_eval.sum(axis=0) > seuil]

data_eval = pd.merge(data_eval, GENE_eval, on = "ID", how = "left")

In [192]:
# VAF

high_risk_genes = ["TP53", "RUNX1", "ASXL1", "DNMT3A", "FLT3"]

vaf_df = maf_df[maf_df["GENE"].isin(high_risk_genes)].groupby("ID").agg(
    VAF_MAX_high_risk = ("VAF", "max")
    # VAF_mean_high_risk = ("VAF", "mean"),
    # Nb_mutation_high_risk = ("VAF", "count")
)

data = pd.merge(data, vaf_df, on = "ID", how = "left")


vaf_df_all = maf_df.groupby("ID").agg(VAF_COUNT_all = ("VAF","count"))

data = pd.merge(data, vaf_df_all, on = "ID", how = "left")


In [193]:
# VAF EVAL

high_risk_genes = ["TP53", "RUNX1", "ASXL1", "DNMT3A", "FLT3"]

vaf_df_eval = maf_eval[maf_eval["GENE"].isin(high_risk_genes)].groupby("ID").agg(
    VAF_MAX_high_risk = ("VAF", "max")
    # VAF_mean_high_risk = ("VAF", "mean"),
    # Nb_mutation_high_risk = ("VAF", "count")
)

data_eval = pd.merge(data_eval, vaf_df_eval, on = "ID", how = "left")


vaf_df_all_eval = maf_eval.groupby("ID").agg(VAF_COUNT_all = ("VAF","count"))

data_eval = pd.merge(data_eval, vaf_df_all_eval, on = "ID", how = "left")

In [194]:
# EFFECT

seuil = 10
effect_df = pd.crosstab(maf_df["ID"], maf_df["EFFECT"])
effect_df = effect_df.loc[:, effect_df.sum(axis = 0) > seuil]

data = pd.merge(data, effect_df, on = "ID", how = "left")

In [195]:
# EFFECT EVAL

effect_df_eval = pd.crosstab(maf_eval["ID"], maf_eval["EFFECT"])
effect_df_eval = effect_df_eval.loc[:, effect_df_eval.sum(axis = 0) > seuil]

data_eval = pd.merge(data_eval, effect_df_eval, on = "ID", how = "left")

# Training

In [196]:
y = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_df)
X = data[data["ID"].isin(target_df["ID"])].copy()

In [197]:
X_eval = data_eval.copy()

In [198]:
common_features = [col for col in X.columns if col in X_eval.columns]

In [199]:
X_train_final = X[common_features]
X_eval_final = X_eval[common_features]

In [ ]:
# features_set = {    
#                      'base_wo_MONOCYTES' : ['BM_BLAST', 'WBC', 'ANC', 'HB', 'PLT',
#        'NOT_MISS_BM_BLAST', 'NOT_MISS_ANC', 'MALE',
#        'FEMALE', 'UNKNOWN_SEX', '46', '20', '13', '11', '2', 'del(5)', '1',
#        '45', '12', '21', '47', '10', '3', '+8', '5', '22', '14', '-7', '4',
#        '7', '15', '33', '17', '25', '23', '6', '31', '8', '9', 'del(20)', '16',
#        '18', '19', '24', '26', '34', '44', '-5', '-18', 'del(7)', '32',
#        'del(12)', '-17', 'del(11)', '+21'],
                    
#                     'base_w_GENE' : ['BM_BLAST', 'WBC', 'ANC', 'HB', 'PLT',
#        'NOT_MISS_BM_BLAST', 'NOT_MISS_ANC', 'NOT_MISS_MONOCYTES', 'MALE',
#        'FEMALE', 'UNKNOWN_SEX', '46', '20', '13', '11', '2', 'del(5)', '1',
#        '45', '12', '21', '47', '10', '3', '+8', '5', '22', '14', '-7', '4',
#        '7', '15', '33', '17', '25', '23', '6', '31', '8', '9', 'del(20)', '16',
#        '18', '19', '24', '26', '34', '44', '-5', '-18', 'del(7)', '32',
#        'del(12)', '-17', 'del(11)', '+21', 'ARID2', 'ASXL1', 'ASXL2', 'ATRX', 'BCOR', 'BCORL1', 'BRAF', 'BRCC3',
#        'CBL', 'CEBPA', 'CREBBP', 'CSF3R', 'CSNK1A1', 'CTCF', 'CUX1', 'DDX41',
#        'DDX54', 'DNMT3A', 'EED', 'EP300', 'ETNK1', 'ETV6', 'EZH2', 'FLT3',
#        'GATA2', 'GNAS', 'GNB1', 'IDH1', 'IDH2', 'IRF1', 'JAK2', 'KDM6A', 'KIT',
#        'KMT2C', 'KMT2D', 'KRAS', 'LUC7L2', 'MGA', 'MLL', 'MPL', 'NF1', 'NFE2',
#        'NPM1', 'NRAS', 'PHF6', 'PPM1D', 'PRPF8', 'PTPN11', 'RAD21', 'RUNX1',
#        'SETBP1', 'SF3B1', 'SH2B3', 'SMC1A', 'SRSF2', 'STAG2', 'STAT3', 'SUZ12',
#        'TET2', 'TP53', 'U2AF1', 'U2AF2', 'WT1', 'ZBTB33', 'ZRSR2'],
       
                    
#                     'base_w_GENE_VAF_EFFECT' : ['BM_BLAST', 'WBC', 'ANC', 'HB', 'PLT',
#        'NOT_MISS_BM_BLAST', 'NOT_MISS_ANC', 'NOT_MISS_MONOCYTES', 'MALE',
#        'FEMALE', 'UNKNOWN_SEX', '46', '20', '13', '11', '2', 'del(5)', '1',
#        '45', '12', '21', '47', '10', '3', '+8', '5', '22', '14', '-7', '4',
#        '7', '15', '33', '17', '25', '23', '6', '31', '8', '9', 'del(20)', '16',
#        '18', '19', '24', '26', '34', '44', '-5', '-18', 'del(7)', '32',
#        'del(12)', '-17', 'del(11)', '+21', 'ARID2', 'ASXL1', 'ASXL2', 'ATRX', 'BCOR', 'BCORL1', 'BRAF', 'BRCC3',
#        'CBL', 'CEBPA', 'CREBBP', 'CSF3R', 'CSNK1A1', 'CTCF', 'CUX1', 'DDX41',
#        'DDX54', 'DNMT3A', 'EED', 'EP300', 'ETNK1', 'ETV6', 'EZH2', 'FLT3',
#        'GATA2', 'GNAS', 'GNB1', 'IDH1', 'IDH2', 'IRF1', 'JAK2', 'KDM6A', 'KIT',
#        'KMT2C', 'KMT2D', 'KRAS', 'LUC7L2', 'MGA', 'MLL', 'MPL', 'NF1', 'NFE2',
#        'NPM1', 'NRAS', 'PHF6', 'PPM1D', 'PRPF8', 'PTPN11', 'RAD21', 'RUNX1',
#        'SETBP1', 'SF3B1', 'SH2B3', 'SMC1A', 'SRSF2', 'STAG2', 'STAT3', 'SUZ12',
#        'TET2', 'TP53', 'U2AF1', 'U2AF2', 'WT1', 'ZBTB33', 'ZRSR2', 'VAF_MAX_high_risk', 'VAF_COUNT_all', 'frameshift_variant', 'inframe_codon_gain',
#        'inframe_codon_loss', 'initiator_codon_change', 'non_synonymous_codon',
#        'splice_site_variant', 'stop_gained']                 
                    
# }

In [201]:
final_features = ['BM_BLAST', 'WBC', 'ANC', 'HB', 'PLT',
       'NOT_MISS_BM_BLAST', 'NOT_MISS_ANC', 'NOT_MISS_MONOCYTES', 'MALE',
       'FEMALE', 'UNKNOWN_SEX', '46', '20', '13', '11', '2', 'del(5)', '1',
       '45', '12', '21', '47', '10', '3', '+8', '5', '22', '14', '-7', '4',
       '7', '15', '33', '17', '25', '23', '6', '31', '8', '9', 'del(20)', '16',
       '18', '19', '24', '26', '34', '44', '-5', '-18', 'del(7)', '32',
       'del(12)', '-17', 'del(11)', '+21', 'ARID2', 'ASXL1', 'ASXL2', 'ATRX', 'BCOR', 'BCORL1', 'BRAF', 'BRCC3',
       'CBL', 'CEBPA', 'CREBBP', 'CSF3R', 'CSNK1A1', 'CTCF', 'CUX1', 'DDX41',
       'DDX54', 'DNMT3A', 'EED', 'EP300', 'ETNK1', 'ETV6', 'EZH2', 'FLT3',
       'GATA2', 'GNAS', 'GNB1', 'IDH1', 'IDH2', 'IRF1', 'JAK2', 'KDM6A', 'KIT',
       'KMT2C', 'KMT2D', 'KRAS', 'LUC7L2', 'MGA', 'MLL', 'MPL', 'NF1', 'NFE2',
       'NPM1', 'NRAS', 'PHF6', 'PPM1D', 'PRPF8', 'PTPN11', 'RAD21', 'RUNX1',
       'SETBP1', 'SF3B1', 'SH2B3', 'SMC1A', 'SRSF2', 'STAG2', 'STAT3', 'SUZ12',
       'TET2', 'TP53', 'U2AF1', 'U2AF2', 'WT1', 'ZBTB33', 'ZRSR2', 'VAF_MAX_high_risk', 'VAF_COUNT_all', 'frameshift_variant', 'inframe_codon_gain',
       'inframe_codon_loss', 'initiator_codon_change', 'non_synonymous_codon',
       'splice_site_variant', 'stop_gained']  

final_features = [col for col in final_features if col in common_features]

In [ ]:
# scores = []

# kf = KFold(n_splits=5, shuffle=True, random_state=11)

# for name, features in features_set.items():
#     if name in ["base_wo_MONOCYTES", "base_w_GENE"]:
#         continue
#     fold_scores = []

#     for train_idx, val_idx in kf.split(X):

#         X_train = X.iloc[train_idx][features]
#         X_val = X.iloc[val_idx][features]

#         y_train = y[train_idx]
#         y_val = y[val_idx]

#         rsf = RandomSurvivalForest(
#             n_estimators=250,
#             min_samples_split=30,
#             min_samples_leaf=15,
#             max_features="sqrt",
#             max_depth = 5,
#             bootstrap=True,
#             n_jobs=-1,
#             random_state=42
#         )

#         rsf.fit(X_train, y_train)

#         train_ci_ipcw = concordance_index_ipcw(y_train, y_train, rsf.predict(X_train), tau = 7)[0]
#         test_ci_ipcw = concordance_index_ipcw(y_train, y_val, rsf.predict(X_val), tau=7)[0]
#         fold_scores.append(test_ci_ipcw)

#     scores.append((name, np.mean(fold_scores)))
#     print(f"{name} — mean C-index IPCW: {np.mean(fold_scores):.4f}")

In [204]:
rsf = RandomSurvivalForest(
            n_estimators=250,
            min_samples_split=30,
            min_samples_leaf=15,
            max_features="sqrt",
            max_depth = 5,
            bootstrap=True,
            n_jobs=-1,
            random_state=42
        )

rsf.fit(X_train_final[final_features], y)

print(concordance_index_ipcw(y, y, rsf.predict(X_train_final[final_features]), tau = 7)[0])



0.7298882004989008


In [ ]:
# scores = []

# name = "finale_features"
# features = final_features

# fold_scores = []

# for train_idx, val_idx in kf.split(X):

#     X_train = X.iloc[train_idx][features]
#     X_val = X.iloc[val_idx][features]

#     y_train = y[train_idx]
#     y_val = y[val_idx]

#     # rsf = RandomSurvivalForest(
#     #     n_estimators=750,         
#     #     min_samples_split=6,     
#     #     min_samples_leaf=2,        
#     #     max_features="sqrt",       
#     #     max_depth=None,           
#     #     bootstrap=True,
#     #     n_jobs=-1,
#     #     random_state=42
#     # )

#     # rsf.fit(X_train, y_train)

#     train_ci_ipcw = concordance_index_ipcw(y_train, y_train, rsf.predict(X_train), tau = 7)[0]
#     test_ci_ipcw = concordance_index_ipcw(y_train, y_val, rsf.predict(X_val), tau=7)[0]
#     fold_scores.append(test_ci_ipcw)

# scores.append((name, np.mean(fold_scores)))
# print(f"{name} — mean C-index IPCW: {np.mean(fold_scores):.4f}")
# print(f"{name} — std C-index IPCW: {np.std(fold_scores):.4f}")

finale_features — mean C-index IPCW: 0.7304
finale_features — std C-index IPCW: 0.0160


# Parameter optimization with Optuna

In [206]:
# def objective(trial):
#     # Définition de l'espace de recherche
#     n_estimators = trial.suggest_int("n_estimators", 200, 1000)
#     min_samples_split = trial.suggest_int("min_samples_split", 2, 50)
#     min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 30)
#     max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])
#     max_depth = trial.suggest_int("max_depth", 3, 30)

#     fold_scores = []

#     for train_idx, val_idx in kf.split(X):
#         X_train, X_val = X.iloc[train_idx][features_set["base_w_GENE_VAF_EFFECT"]], X.iloc[val_idx][features_set["base_w_GENE_VAF_EFFECT"]]
#         y_train, y_val = y[train_idx], y[val_idx]

#         rsf = RandomSurvivalForest(
#             n_estimators=n_estimators,
#             min_samples_split=min_samples_split,
#             min_samples_leaf=min_samples_leaf,
#             max_features=max_features,
#             max_depth=max_depth,
#             bootstrap=True,
#             n_jobs=-1,
#             random_state=42
#         )

#         rsf.fit(X_train, y_train)
#         ci = concordance_index_ipcw(y_train, y_val, rsf.predict(X_val), tau=7)[0]
#         fold_scores.append(ci)

#     mean_ci = np.mean(fold_scores)
#     std_ci = np.std(fold_scores)


#     return mean_ci - 0.5 * std_ci  

In [207]:
# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=50)

# print("Best params:", study.best_params)
# print("Best score:", study.best_value)

# Prediction

In [208]:
prediction_on_test_set = rsf.predict(X_eval_final[final_features])

In [ ]:
submission = pd.DataFrame({
    "risk_score": prediction_on_test_set
}, index=df_eval['ID'])

submission.to_csv('./submission.csv', index_label="ID")